# TUNAS Digital — Chatbot Edukasi Perlindungan Anak
**Setup & Jalankan Streamlit di Google Colab**

> Pastikan kamu sudah menyimpan API key berikut di **Colab Secrets** (ikon 🔑 di sidebar kiri):
> - `GEMINI` → Google Gemini API Key (dari [aistudio.google.com](https://aistudio.google.com/app/apikey))
> - `EXA` → Exa API Key (dari [exa.ai](https://exa.ai))
> - `NGROK_TOKEN` → ngrok Auth Token (dari [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken))

## Sel 1 — Install Semua Dependencies

In [24]:
# Install semua library yang dibutuhkan
!pip install -q streamlit pyngrok google-genai exa-py PyPDF2
!pip install -q langchain langchain-google-genai langchain-community chromadb langchain-text-splitters
!pip install -q -U langchain-google-genai   # ← tambah -U untuk upgrade ke versi terbaru
print(' Semua library berhasil diinstall!')

 Semua library berhasil diinstall!


In [25]:
!pip install -q -U google-generativeai

## Sel 2 — Tulis File Streamlit App

In [28]:
%%writefile streamlit_tunas_app.py
import streamlit as st
import json
import os
import re
import time
import tempfile
import PyPDF2

# ─────────────────────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title='🌱 TUNAS Digital — Perlindungan Anak di Ruang Digital',
    layout='wide',
    initial_sidebar_state='expanded',
)

# ─────────────────────────────────────────────────────────────
# CUSTOM CSS
# ─────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700;800&family=Quicksand:wght@500;600;700&display=swap');
html, body, [class*="css"] { font-family: 'Nunito', sans-serif; }
.hero-header {
    background: linear-gradient(135deg, #1a6b3a 0%, #2d9e5f 50%, #4ecb87 100%);
    border-radius: 18px; padding: 28px 36px; margin-bottom: 24px;
    box-shadow: 0 8px 32px rgba(30,120,70,0.18);
    display: flex; align-items: center; gap: 20px;
}
.hero-title {
    font-family: 'Quicksand', sans-serif; font-size: 2rem;
    font-weight: 700; color: #fff; margin: 0; line-height: 1.2;
}
.hero-sub { color: #c8f5de; font-size: 0.95rem; margin-top: 4px; }
.source-badge {
    display: inline-block; background: #e8f8f0;
    border: 1px solid #7dd9a8; color: #1a6b3a;
    border-radius: 20px; padding: 2px 12px;
    font-size: 0.78rem; font-weight: 600; margin: 4px 3px 0 0;
}
.info-card {
    background: #f0faf4; border-left: 4px solid #2d9e5f;
    border-radius: 0 12px 12px 0; padding: 12px 16px;
    margin: 10px 0; font-size: 0.9rem; color: #1a3a28;
}
.pill-green {
    background: #dcfce7; color: #15803d; border-radius: 20px;
    padding: 3px 12px; font-size: 0.8rem; font-weight: 700; display: inline-block;
}
.pill-orange {
    background: #fff7ed; color: #c2410c; border-radius: 20px;
    padding: 3px 12px; font-size: 0.8rem; font-weight: 700; display: inline-block;
}
section[data-testid='stSidebar'] {
    background: linear-gradient(180deg, #dff0e6 0%, #c5e8d4 100%);
}
section[data-testid='stSidebar'] label,
section[data-testid='stSidebar'] p,
section[data-testid='stSidebar'] span,
section[data-testid='stSidebar'] h1,
section[data-testid='stSidebar'] h2,
section[data-testid='stSidebar'] h3,
section[data-testid='stSidebar'] .stMarkdown { color: #0d4a24 !important; }
section[data-testid='stSidebar'] input,
section[data-testid='stSidebar'] textarea {
    color: #111111 !important;
    background: rgba(255,255,255,0.92) !important;
    border-radius: 8px !important;
}
section[data-testid='stSidebar'] [data-testid='stFileUploader'] span,
section[data-testid='stSidebar'] [data-testid='stFileUploader'] p,
section[data-testid='stSidebar'] [data-testid='stFileUploaderDropzoneInstructions'] * {
    color: #111111 !important;
}
section[data-testid='stSidebar'] [data-testid='stFileUploader'] {
    background: rgba(255,255,255,0.92) !important;
    border-radius: 10px !important;
    padding: 4px !important;
}
section[data-testid='stSidebar'] .stButton button {
    background: #2d9e5f !important; color: #ffffff !important;
    border: none !important; border-radius: 10px !important;
    font-weight: 700 !important; width: 100%;
}
section[data-testid='stSidebar'] .stButton button:hover {
    background: #1a6b3a !important;
}
#MainMenu, footer { visibility: hidden; }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────
# HERO HEADER
# ─────────────────────────────────────────────────────────────
st.markdown("""
<div class='hero-header'>
    <div style='font-size:3rem;'>🌱</div>
    <div>
        <div class='hero-title'>TUNAS Digital — Asisten Edukasi Orang Tua</div>
        <div class='hero-sub'>Sosialisasi PP Perlindungan Anak di Ruang Digital · Berbasis PP Tunas & Pencarian Realtime</div>
    </div>
</div>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown('## ⚙️ Konfigurasi')
    st.markdown('---')
    google_api_key = st.text_input('🔑 Google Gemini API Key', type='password', placeholder='AIza...')
    exa_api_key    = st.text_input('🔍 Exa API Key', type='password', placeholder='exa-...')
    st.markdown('---')
    st.markdown('## 📄 Dokumen RAG (PDF)')
    uploaded_pdfs = st.file_uploader(
        'Upload PDF PP Tunas / Regulasi',
        type=['pdf'],
        accept_multiple_files=True,
        help='Upload satu atau lebih dokumen PDF.'
    )
    build_rag_btn = st.button('🔨 Bangun / Perbarui Index RAG', use_container_width=True)
    if 'rag_built' in st.session_state and st.session_state.rag_built:
        st.markdown('<span class="pill-green">✅ RAG Aktif</span>', unsafe_allow_html=True)
        if 'pdf_names' in st.session_state:
            for name in st.session_state.pdf_names:
                st.markdown(f"<span style='font-size:0.82rem; color:#0d4a24;'>📎 {name}</span>", unsafe_allow_html=True)
    else:
        st.markdown('<span class="pill-orange">⏳ Belum ada index RAG</span>', unsafe_allow_html=True)
    st.markdown('---')
    st.markdown('## 🎯 Mode Pencarian')
    use_exa = st.toggle('Aktifkan Exa Web Search', value=True)
    use_rag = st.toggle('Aktifkan RAG (PDF Lokal)', value=True)
    st.markdown('---')
    if st.button('🗑️ Reset Percakapan', use_container_width=True):
        st.session_state.messages = []
        st.rerun()
    st.markdown('---')
    st.markdown("""
    <div style='font-size:0.78rem; color:#1a4a2e; line-height:1.6;'>
    <b>📌 Tentang Aplikasi</b><br>
    Dirancang untuk membantu orang tua memahami hak & perlindungan anak di ruang digital
    berdasarkan regulasi Indonesia.<br><br>
    <b>Sumber:</b> PP Tunas, UU ITE, KPAI
    </div>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────
# GUARD — API KEYS
# ─────────────────────────────────────────────────────────────
if not google_api_key or not exa_api_key:
    st.markdown("""
    <div class='info-card'>
    🔐 <b>Masukkan API Key terlebih dahulu</b> di sidebar kiri untuk mulai menggunakan chatbot.<br><br>
    Butuh API Key?
    <ul>
        <li>Google Gemini: <a href='https://aistudio.google.com/app/apikey' target='_blank'>aistudio.google.com</a></li>
        <li>Exa: <a href='https://exa.ai' target='_blank'>exa.ai</a></li>
    </ul>
    </div>
    """, unsafe_allow_html=True)
    st.stop()

# ─────────────────────────────────────────────────────────────
# LAZY IMPORTS
# ─────────────────────────────────────────────────────────────
try:
    from google import genai
    from google.genai import types as gtypes
    from exa_py import Exa
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from langchain_community.vectorstores import Chroma
except ImportError as e:
    st.error(f'❌ Library belum terinstall: {e}')
    st.stop()

# ─────────────────────────────────────────────────────────────
# INIT CLIENTS
# ─────────────────────────────────────────────────────────────
@st.cache_resource
def get_gemini_client(api_key):
    return genai.Client(api_key=api_key)

@st.cache_resource
def get_exa_client(api_key):
    return Exa(api_key=api_key)

gemini_client = get_gemini_client(google_api_key)
exa_client    = get_exa_client(exa_api_key)

# ─────────────────────────────────────────────────────────────
# RAG FUNCTIONS
# ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_text_from_pdf(file_bytes) -> str:
    reader = PyPDF2.PdfReader(file_bytes)
    return '\n\n'.join(page.extract_text() or '' for page in reader.pages)

def get_embedding(text: str, api_key: str) -> list:
    import google.generativeai as genai_legacy
    import time
    genai_legacy.configure(api_key=api_key)

    # Bersihkan lebih agresif sebelum kirim
    text = text.strip()
    text = text[:2000]  # batasi max 2000 karakter per chunk
    if not text:
        return [0.0] * 768  # return zero vector jika kosong

    for attempt in range(3):  # retry 3x
        try:
            result = genai_legacy.embed_content(
                model='models/text-embedding-004',
                content=text,
            )
            time.sleep(0.3)  # jeda antar request
            return result['embedding']
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)  # exponential backoff
                continue
            # Jika tetap gagal, return zero vector & lanjut
            st.warning(f'⚠️ Skip chunk gagal di-embed: {str(e)[:60]}')
            return [0.0] * 768

def build_rag_index(pdf_files, api_key):
    import google.generativeai as genai_legacy
    import chromadb
    genai_legacy.configure(api_key=api_key)

    splitter  = RecursiveCharacterTextSplitter(
        chunk_size=400,      # ← perkecil dari 600
        chunk_overlap=80,
    )
    all_texts = []
    for pdf in pdf_files:
        raw    = extract_text_from_pdf(pdf)
        raw    = clean_text(raw)
        chunks = splitter.split_text(raw)
        all_texts.extend(chunks)

    # Filter: minimal 30 karakter, max 2000 karakter
    all_texts = [
        t.strip() for t in all_texts
        if 30 < len(t.strip()) < 2000
    ]

    st.info(f'📊 Total {len(all_texts)} chunk akan di-embed...')
    embeddings = []
    failed = 0
    for i, chunk in enumerate(all_texts):
        emb = get_embedding(chunk, api_key)
        embeddings.append(emb)
        if (i + 1) % 10 == 0:
            st.info(f'📊 Progress: {i+1}/{len(all_texts)} chunk...')

    # Hapus chunk dengan zero vector (yang gagal)
    valid = [
        (t, e) for t, e in zip(all_texts, embeddings)
        if any(v != 0.0 for v in e)
    ]
    if not valid:
        st.error('❌ Semua chunk gagal di-embed.')
        return None
    all_texts, embeddings = zip(*valid)
    all_texts  = list(all_texts)
    embeddings = list(embeddings)

    persist_dir = './chroma_tunas_db'
    chroma_client = chromadb.PersistentClient(path=persist_dir)
    try:
        chroma_client.delete_collection('tunas_docs')
    except:
        pass

    collection = chroma_client.create_collection('tunas_docs')
    # Tambah dalam batch kecil agar tidak timeout
    batch_size = 50
    for i in range(0, len(all_texts), batch_size):
        batch_texts = all_texts[i:i+batch_size]
        batch_embs  = embeddings[i:i+batch_size]
        batch_ids   = [f'doc_{i+j}' for j in range(len(batch_texts))]
        collection.add(
            documents=batch_texts,
            embeddings=batch_embs,
            ids=batch_ids,
        )

    st.session_state.chroma_collection = collection
    return collection

def retrieve_context(query: str, collection, k=6) -> str:
    import google.generativeai as genai_legacy
    query_emb = genai_legacy.embed_content(
        model='models/text-embedding-004',
        content=query,
        task_type='retrieval_query',
    )['embedding']

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=k,
    )
    docs = results['documents'][0]
    return '\n\n---\n\n'.join(docs)

# ─────────────────────────────────────────────────────────────
# BUILD RAG BUTTON HANDLER
# ─────────────────────────────────────────────────────────────
if build_rag_btn:
    if not uploaded_pdfs:
        st.warning('⚠️ Upload setidaknya satu file PDF terlebih dahulu.')
    else:
        with st.spinner('🔨 Membangun index RAG dari PDF...'):
            db = build_rag_index(uploaded_pdfs, google_api_key)
            st.session_state.rag_db    = db
            st.session_state.rag_built = True
            st.session_state.pdf_names = [f.name for f in uploaded_pdfs]
        st.success('✅ Index RAG berhasil dibangun!')
        st.rerun()

# ─────────────────────────────────────────────────────────────
# EXA SEARCH
# ─────────────────────────────────────────────────────────────
def exa_search(query):
    try:
        results = exa_client.search_and_contents(
            query=query, type='auto', num_results=3,
            text={'max_characters': 1800},
        )
        output = []
        for r in results.results:
            output.append({'title': r.title, 'url': r.url, 'text': r.text})
        return json.dumps(output, ensure_ascii=False, indent=2)
    except Exception as e:
        return f'ERROR: {str(e)}'

# ─────────────────────────────────────────────────────────────
# TOOL DECLARATION
# ─────────────────────────────────────────────────────────────
exa_tool_decl = gtypes.Tool(
    function_declarations=[
        gtypes.FunctionDeclaration(
            name='exa_search_and_contents',
            description='Cari informasi terkini dari internet terkait perlindungan anak, regulasi digital, berita KPAI, Kominfo, atau isu keamanan online anak.',
            parameters={
                'type': 'OBJECT',
                'properties': {
                    'query': {'type': 'STRING', 'description': 'Query pencarian dalam Bahasa Indonesia atau Inggris'}
                },
                'required': ['query'],
            },
        )
    ]
)

# ─────────────────────────────────────────────────────────────
# SYSTEM PROMPT
# ─────────────────────────────────────────────────────────────
def build_system_prompt(rag_context=''):
    rag_section = ''
    if rag_context:
        rag_section = f"""
## KONTEKS DARI DOKUMEN REGULASI (PP TUNAS & DOKUMEN TERKAIT):
{rag_context}
Gunakan konteks di atas sebagai acuan utama jika relevan dengan pertanyaan.
---"""
    return f"""
Kamu adalah **TUNAS Assistant**, asisten edukasi khusus untuk orang tua Indonesia
terkait perlindungan anak di ruang digital.
{rag_section}
## PERAN & KEPRIBADIAN:
- Ramah, sabar, dan mudah dipahami oleh orang tua awam teknologi
- Gunakan bahasa Indonesia yang hangat dan tidak menghakimi
- Berikan informasi berdasarkan regulasi resmi Indonesia (PP Tunas, UU ITE, Permendikbud, dll)
- Selalu berikan langkah praktis yang bisa langsung diterapkan orang tua

## ATURAN PENGGUNAAN TOOL:
- Untuk pertanyaan tentang berita terbaru, kasus terkini, aplikasi baru, atau tren media sosial anak:
  **WAJIB** gunakan tool `exa_search_and_contents`
- Untuk pertanyaan tentang isi regulasi/hukum: prioritaskan konteks RAG di atas
- Selalu sebutkan sumber informasi (URL jika dari web, nama dokumen jika dari RAG)

## TOPIK UTAMA:
1. Isi dan makna PP Tunas & regulasi perlindungan anak digital
2. Hak digital anak dan perlindungan data pribadi anak
3. Ancaman online: cyberbullying, grooming, konten negatif, kecanduan gadget
4. Cara orang tua mendampingi anak di internet
5. Mekanisme pelaporan ke KPAI, Kominfo, Polri
6. Parental control & pengaturan privasi anak

## FORMAT JAWABAN:
- Mulai dengan ringkasan singkat (2-3 kalimat)
- Gunakan poin-poin untuk kejelasan
- Akhiri dengan **saran praktis** untuk orang tua
- Sertakan dasar hukum dan sumber jika relevan
"""

# ─────────────────────────────────────────────────────────────
# SESSION STATE
# ─────────────────────────────────────────────────────────────
if 'messages' not in st.session_state:
    st.session_state.messages = []
if 'rag_built' not in st.session_state:
    st.session_state.rag_built = False

# ─────────────────────────────────────────────────────────────
# QUICK PROMPTS
# ─────────────────────────────────────────────────────────────
if not st.session_state.messages:
    st.markdown('### 💬 Mulai dari pertanyaan ini:')
    quick_prompts = [
        'Apa itu PP Tunas dan apa manfaatnya untuk anak?',
        'Anak saya kena cyberbullying, apa yang harus saya lakukan?',
        'Berapa usia minimal anak boleh punya media sosial?',
        'Bagaimana cara mengatur privasi akun anak di TikTok/Instagram?',
        'Kemana saya harus melapor jika anak jadi korban penipuan online?',
        'Apa bahaya kecanduan game online dan cara mengatasinya?',
    ]
    cols = st.columns(3)
    for i, qp in enumerate(quick_prompts):
        with cols[i % 3]:
            if st.button(qp, use_container_width=True, key=f'qp_{i}'):
                st.session_state.messages.append({'role': 'user', 'content': qp})
                st.rerun()

# ─────────────────────────────────────────────────────────────
# DISPLAY HISTORY
# ─────────────────────────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])
        if msg.get('sources'):
            st.markdown('**🔗 Sumber:**')
            for src in msg['sources']:
                st.markdown(f'<span class="source-badge">🌐 {src}</span>', unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────
# CHAT INPUT
# ─────────────────────────────────────────────────────────────
user_input = st.chat_input('Tanyakan tentang perlindungan anak di ruang digital...')

if user_input:
    st.session_state.messages.append({'role': 'user', 'content': user_input})
    with st.chat_message('user'):
        st.markdown(user_input)

    with st.chat_message('assistant'):
        status_box = st.empty()

        rag_context = ''
        if use_rag and st.session_state.rag_built and 'rag_db' in st.session_state:
            status_box.info('📄 Mencari di dokumen regulasi...')
            rag_context = retrieve_context(user_input, st.session_state.rag_db)

        system_prompt = build_system_prompt(rag_context)
        status_box.info('🤔 Menganalisis pertanyaan...')

        tools_list = [exa_tool_decl] if use_exa else []

        first_response = gemini_client.models.generate_content(
            model='gemini-2.5-flash-lite',
            contents=user_input,
            config=gtypes.GenerateContentConfig(
                system_instruction=system_prompt,
                tools=tools_list if tools_list else None,
            ),
        )

        candidate   = first_response.candidates[0]
        parts       = candidate.content.parts
        final_text  = ''
        sources     = []
        tool_called = False

        for part in parts:
            if hasattr(part, 'function_call') and part.function_call:
                tool_called = True
                fn_name = part.function_call.name
                args    = dict(part.function_call.args)
                query   = args.get('query', user_input)
                status_box.info(f'🔍 Mencari di web: *{query}*')
                tool_result = exa_search(query)
                try:
                    results_json = json.loads(tool_result)
                    sources = [r['url'] for r in results_json if 'url' in r]
                except:
                    pass
                status_box.info('✍️ Merangkum jawaban...')
                second_response = gemini_client.models.generate_content(
                    model='gemini-2.5-flash-lite',
                    contents=[
                        user_input,
                        gtypes.Content(role='model', parts=[part]),
                        gtypes.Content(
                            role='tool',
                            parts=[
                                gtypes.Part.from_function_response(
                                    name=fn_name,
                                    response={'result': tool_result},
                                )
                            ],
                        ),
                    ],
                    config=gtypes.GenerateContentConfig(system_instruction=system_prompt),
                )
                final_text = second_response.text

        if not tool_called:
            final_text = first_response.text

        status_box.empty()
        st.markdown(final_text)

        if sources:
            st.markdown('**🔗 Sumber Web:**')
            for url in sources:
                st.markdown(f'<span class="source-badge">🌐 {url}</span>', unsafe_allow_html=True)
        if rag_context:
            st.markdown('<span class="source-badge">📄 PP Tunas (Dokumen Lokal)</span>', unsafe_allow_html=True)

    st.session_state.messages.append({
        'role':    'assistant',
        'content': final_text,
        'sources': sources,
    })


Overwriting streamlit_tunas_app.py


In [ ]:
# %%writefile streamlit_tunas_app.py
# import streamlit as st
# import json
# import os
# import time
# import tempfile

# # ─────────────────────────────────────────────────────────────
# # PAGE CONFIG
# # ─────────────────────────────────────────────────────────────
# st.set_page_config(
#     page_title='🌱 TUNAS Digital — Perlindungan Anak di Ruang Digital',
#     layout='wide',
#     initial_sidebar_state='expanded',
# )

# # ─────────────────────────────────────────────────────────────
# # CUSTOM CSS
# # ─────────────────────────────────────────────────────────────
# st.markdown("""
# <style>
# @import url('https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700;800&family=Quicksand:wght@500;600;700&display=swap');
# html, body, [class*="css"] { font-family: 'Nunito', sans-serif; }
# .hero-header {
#     background: linear-gradient(135deg, #1a6b3a 0%, #2d9e5f 50%, #4ecb87 100%);
#     border-radius: 18px; padding: 28px 36px; margin-bottom: 24px;
#     box-shadow: 0 8px 32px rgba(30,120,70,0.18);
#     display: flex; align-items: center; gap: 20px;
# }
# .hero-title {
#     font-family: 'Quicksand', sans-serif; font-size: 2rem;
#     font-weight: 700; color: #fff; margin: 0; line-height: 1.2;
# }
# .hero-sub { color: #c8f5de; font-size: 0.95rem; margin-top: 4px; }
# .source-badge {
#     display: inline-block; background: #e8f8f0;
#     border: 1px solid #7dd9a8; color: #1a6b3a;
#     border-radius: 20px; padding: 2px 12px;
#     font-size: 0.78rem; font-weight: 600; margin: 4px 3px 0 0;
# }
# .info-card {
#     background: #f0faf4; border-left: 4px solid #2d9e5f;
#     border-radius: 0 12px 12px 0; padding: 12px 16px;
#     margin: 10px 0; font-size: 0.9rem; color: #1a3a28;
# }
# .pill-green {
#     background: #dcfce7; color: #9bf2bc; border-radius: 20px;
#     padding: 3px 12px; font-size: 0.8rem; font-weight: 700; display: inline-block;
# }
# .pill-orange {
#     background: #fff7ed; color: #c2410c; border-radius: 20px;
#     padding: 3px 12px; font-size: 0.8rem; font-weight: 700; display: inline-block;
# }
# section[data-testid='stSidebar'] {
#     background: linear-gradient(180deg, #dff0e6 0%);
# }
# section[data-testid='stSidebar'] * { color: #1a3a6b !important; }
# section[data-testid='stSidebar'] .stButton button {
#     background: #4ecb87 !important; color: #0d4a24 !important;
#     border: none !important; border-radius: 10px !important;
#     font-weight: 700 !important; width: 100%;
# }
# #MainMenu, footer { visibility: hidden; }
# </style>
# """, unsafe_allow_html=True)

# # ─────────────────────────────────────────────────────────────
# # HERO HEADER
# # ─────────────────────────────────────────────────────────────
# st.markdown("""
# <div class='hero-header'>
#     <div style='font-size:3rem;'>🌱</div>
#     <div>
#         <div class='hero-title'>TUNAS Digital — Asisten Edukasi Orang Tua</div>
#         <div class='hero-sub'>Sosialisasi PP Perlindungan Anak di Ruang Digital · Berbasis PP Tunas & Pencarian Realtime</div>
#     </div>
# </div>
# """, unsafe_allow_html=True)

# # ─────────────────────────────────────────────────────────────
# # SIDEBAR
# # ─────────────────────────────────────────────────────────────
# with st.sidebar:
#     st.markdown('## ⚙️ Konfigurasi')
#     st.markdown('---')
#     google_api_key = st.text_input('🔑 Google Gemini API Key', type='password', placeholder='AIza...')
#     exa_api_key    = st.text_input('🔍 Exa API Key', type='password', placeholder='exa-...')
#     st.markdown('---')
#     st.markdown('## 📄 Dokumen RAG (PDF)')
#     uploaded_pdfs = st.file_uploader(
#         'Upload PDF PP Tunas / Regulasi',
#         type=['pdf'],
#         accept_multiple_files=True,
#         help='Upload satu atau lebih dokumen PDF. Dokumen akan diindeks sebagai basis pengetahuan chatbot.'
#     )
#     build_rag_btn = st.button('Bangun / Perbarui Index RAG', use_container_width=True)
#     if 'rag_built' in st.session_state and st.session_state.rag_built:
#         st.markdown('<span class="pill-green">✅ RAG Aktif</span>', unsafe_allow_html=True)
#         if 'pdf_names' in st.session_state:
#             for name in st.session_state.pdf_names:
#                 st.markdown(f"<span style='font-size:0.82rem;'>📎 {name}</span>", unsafe_allow_html=True)
#     else:
#         st.markdown('<span class="pill-orange">⏳ Belum ada index RAG</span>', unsafe_allow_html=True)
#     st.markdown('---')
#     st.markdown('## Mode Pencarian')
#     use_exa = st.toggle('Aktifkan Exa Web Search', value=True)
#     use_rag = st.toggle('Aktifkan RAG (PDF Lokal)', value=True)
#     st.markdown('---')
#     if st.button('Reset Percakapan', use_container_width=True):
#         st.session_state.messages = []
#         st.rerun()
#     st.markdown('---')
#     st.markdown("""
#     <div style='font-size:0.78rem; opacity:0.75; line-height:1.6;'>
#     <b>📌 Tentang Aplikasi</b><br>
#     Dirancang untuk membantu orang tua memahami hak & perlindungan anak di ruang digital
#     berdasarkan regulasi Indonesia.<br><br>
#     <b>Sumber:</b> PP Tunas, UU ITE, KPAI
#     </div>
#     """, unsafe_allow_html=True)

# # ─────────────────────────────────────────────────────────────
# # GUARD — API KEYS
# # ─────────────────────────────────────────────────────────────
# if not google_api_key or not exa_api_key:
#     st.markdown("""
#     <div class='info-card'>
#      <b>Masukkan API Key terlebih dahulu</b> di sidebar kiri untuk mulai menggunakan chatbot.<br><br>
#     Butuh API Key?
#     <ul>
#         <li>Google Gemini: <a href='https://aistudio.google.com/app/apikey' target='_blank'>aistudio.google.com</a></li>
#         <li>Exa: <a href='https://exa.ai' target='_blank'>exa.ai</a></li>
#     </ul>
#     </div>
#     """, unsafe_allow_html=True)
#     st.stop()

# # ─────────────────────────────────────────────────────────────
# # LAZY IMPORTS
# # ─────────────────────────────────────────────────────────────
# try:
#     from google import genai
#     from google.genai import types as gtypes
#     from exa_py import Exa
#     import PyPDF2
#     from langchain_text_splitters import RecursiveCharacterTextSplitter
#     from langchain_google_genai import GoogleGenerativeAIEmbeddings
#     from langchain_community.vectorstores import Chroma
# except ImportError as e:
#     st.error(f'❌ Library belum terinstall: {e}')
#     st.stop()

# # ─────────────────────────────────────────────────────────────
# # INIT CLIENTS
# # ─────────────────────────────────────────────────────────────
# @st.cache_resource
# def get_gemini_client(api_key):
#     return genai.Client(api_key=api_key)

# @st.cache_resource
# def get_exa_client(api_key):
#     return Exa(api_key=api_key)

# gemini_client = get_gemini_client(google_api_key)
# exa_client    = get_exa_client(exa_api_key)

# # ─────────────────────────────────────────────────────────────
# # RAG FUNCTIONS
# # ─────────────────────────────────────────────────────────────
# def extract_text_from_pdf(file_bytes):
#     reader = PyPDF2.PdfReader(file_bytes)
#     return '\n\n'.join(page.extract_text() or '' for page in reader.pages)

# def build_rag_index(pdf_files, api_key):
#     splitter  = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)
#     all_texts = []
#     for pdf in pdf_files:
#         raw    = extract_text_from_pdf(pdf)
#         raw    = clean_text(raw)
#         chunks = splitter.split_text(raw)
#         all_texts.extend(chunks)

#     all_texts = [t for t in all_texts if len(t.strip()) > 20]

#     embeddings = GoogleGenerativeAIEmbeddings(
#         model='models/text-embedding-004',  # ← fix di sini
#         google_api_key=api_key,
#     )
#     persist_dir = './chroma_tunas_db'
#     db = Chroma.from_texts(all_texts, embeddings, persist_directory=persist_dir)
#     db.persist()

#     return Chroma(
#         persist_directory=persist_dir,
#         embedding_function=GoogleGenerativeAIEmbeddings(
#             model='models/text-embedding-004',  # ← dan di sini
#             google_api_key=api_key,
#         )
#     )

# def retrieve_context(query, db, k=6):
#     retriever = db.as_retriever(search_kwargs={'k': k})
#     docs = retriever.get_relevant_documents(query)
#     return '\n\n---\n\n'.join(d.page_content for d in docs)

# # ─────────────────────────────────────────────────────────────
# # BUILD RAG BUTTON HANDLER
# # ─────────────────────────────────────────────────────────────
# if build_rag_btn:
#     if not uploaded_pdfs:
#         st.warning('Upload setidaknya satu file PDF terlebih dahulu.')
#     else:
#         with st.spinner('Membangun index RAG dari PDF...'):
#             db = build_rag_index(uploaded_pdfs, google_api_key)
#             st.session_state.rag_db    = db
#             st.session_state.rag_built = True
#             st.session_state.pdf_names = [f.name for f in uploaded_pdfs]
#         st.success('Index RAG berhasil dibangun!')
#         st.rerun()

# # ─────────────────────────────────────────────────────────────
# # EXA SEARCH
# # ─────────────────────────────────────────────────────────────
# def exa_search(query):
#     try:
#         results = exa_client.search_and_contents(
#             query=query, type='auto', num_results=3,
#             text={'max_characters': 1800},
#         )
#         output = []
#         for r in results.results:
#             output.append({'title': r.title, 'url': r.url, 'text': r.text})
#         return json.dumps(output, ensure_ascii=False, indent=2)
#     except Exception as e:
#         return f'ERROR: {str(e)}'

# # ─────────────────────────────────────────────────────────────
# # TOOL DECLARATION
# # ─────────────────────────────────────────────────────────────
# exa_tool_decl = gtypes.Tool(
#     function_declarations=[
#         gtypes.FunctionDeclaration(
#             name='exa_search_and_contents',
#             description='Cari informasi terkini dari internet terkait perlindungan anak, regulasi digital, berita KPAI, Kominfo, atau isu keamanan online anak.',
#             parameters={
#                 'type': 'OBJECT',
#                 'properties': {
#                     'query': {'type': 'STRING', 'description': 'Query pencarian dalam Bahasa Indonesia atau Inggris'}
#                 },
#                 'required': ['query'],
#             },
#         )
#     ]
# )

# # ─────────────────────────────────────────────────────────────
# # SYSTEM PROMPT
# # ─────────────────────────────────────────────────────────────
# def build_system_prompt(rag_context=''):
#     rag_section = ''
#     if rag_context:
#         rag_section = f"""
# ## KONTEKS DARI DOKUMEN REGULASI (PP TUNAS & DOKUMEN TERKAIT):
# {rag_context}
# Gunakan konteks di atas sebagai acuan utama jika relevan dengan pertanyaan.
# ---"""
#     return f"""
# Kamu adalah **TUNAS Assistant**, asisten edukasi khusus untuk orang tua Indonesia
# terkait perlindungan anak di ruang digital.
# {rag_section}
# ## PERAN & KEPRIBADIAN:
# - Ramah, sabar, dan mudah dipahami oleh orang tua awam teknologi
# - Gunakan bahasa Indonesia yang hangat dan tidak menghakimi
# - Berikan informasi berdasarkan regulasi resmi Indonesia (PP Tunas, UU ITE, Permendikbud, dll)
# - Selalu berikan langkah praktis yang bisa langsung diterapkan orang tua

# ## ATURAN PENGGUNAAN TOOL:
# - Untuk pertanyaan tentang berita terbaru, kasus terkini, aplikasi baru, atau tren media sosial anak:
#   **WAJIB** gunakan tool `exa_search_and_contents`
# - Untuk pertanyaan tentang isi regulasi/hukum: prioritaskan konteks RAG di atas
# - Selalu sebutkan sumber informasi (URL jika dari web, nama dokumen jika dari RAG)

# ## TOPIK UTAMA:
# 1. Isi dan makna PP Tunas & regulasi perlindungan anak digital
# 2. Hak digital anak dan perlindungan data pribadi anak
# 3. Ancaman online: cyberbullying, grooming, konten negatif, kecanduan gadget
# 4. Cara orang tua mendampingi anak di internet
# 5. Mekanisme pelaporan ke KPAI, Kominfo, Polri
# 6. Parental control & pengaturan privasi anak

# ## FORMAT JAWABAN:
# - Mulai dengan ringkasan singkat (2-3 kalimat)
# - Gunakan poin-poin untuk kejelasan
# - Akhiri dengan **saran praktis** untuk orang tua
# - Sertakan dasar hukum dan sumber jika relevan
# """
# # ─────────────────────────────────────────────────────────────
# # RAG FUNCTIONS
# # ─────────────────────────────────────────────────────────────
# import re

# def clean_text(text: str) -> str:
#     """Hapus emoji & karakter non-ASCII agar bisa di-embed."""
#     text = text.encode('ascii', 'ignore').decode('ascii')
#     text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# def extract_text_from_pdf(file_bytes):
#     reader = PyPDF2.PdfReader(file_bytes)
#     return '\n\n'.join(page.extract_text() or '' for page in reader.pages)

# def build_rag_index(pdf_files, api_key):
#     splitter  = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)
#     all_texts = []
#     for pdf in pdf_files:
#         raw    = extract_text_from_pdf(pdf)
#         raw    = clean_text(raw)
#         chunks = splitter.split_text(raw)
#         all_texts.extend(chunks)

#     all_texts = [t for t in all_texts if len(t.strip()) > 20]

#     embeddings = GoogleGenerativeAIEmbeddings(
#         model='models/text-embedding-004',
#         google_api_key=api_key,
#     )
#     persist_dir = './chroma_tunas_db'
#     db = Chroma.from_texts(all_texts, embeddings, persist_directory=persist_dir)
#     db.persist()

#     return Chroma(
#         persist_directory=persist_dir,
#         embedding_function=GoogleGenerativeAIEmbeddings(
#             model='models/text-embedding-004',
#             google_api_key=api_key,
#         )
#     )

# def retrieve_context(query, db, k=6):
#     retriever = db.as_retriever(search_kwargs={'k': k})
#     docs = retriever.get_relevant_documents(query)
#     return '\n\n---\n\n'.join(d.page_content for d in docs)
# # ─────────────────────────────────────────────────────────────
# # SESSION STATE
# # ─────────────────────────────────────────────────────────────
# if 'messages' not in st.session_state:
#     st.session_state.messages = []
# if 'rag_built' not in st.session_state:
#     st.session_state.rag_built = False

# # ─────────────────────────────────────────────────────────────
# # QUICK PROMPTS
# # ─────────────────────────────────────────────────────────────
# if not st.session_state.messages:
#     st.markdown('### 💬 Mulai dari pertanyaan ini:')
#     quick_prompts = [
#         'Apa itu PP Tunas dan apa manfaatnya untuk anak?',
#         'Anak saya kena cyberbullying, apa yang harus saya lakukan?',
#         'Berapa usia minimal anak boleh punya media sosial?',
#         'Bagaimana cara mengatur privasi akun anak di TikTok/Instagram?',
#         'Kemana saya harus melapor jika anak jadi korban penipuan online?',
#         'Apa bahaya kecanduan game online dan cara mengatasinya?',
#     ]
#     cols = st.columns(3)
#     for i, qp in enumerate(quick_prompts):
#         with cols[i % 3]:
#             if st.button(qp, use_container_width=True, key=f'qp_{i}'):
#                 st.session_state.messages.append({'role': 'user', 'content': qp})
#                 st.rerun()

# # ─────────────────────────────────────────────────────────────
# # DISPLAY HISTORY
# # ─────────────────────────────────────────────────────────────
# for msg in st.session_state.messages:
#     with st.chat_message(msg['role']):
#         st.markdown(msg['content'])
#         if msg.get('sources'):
#             st.markdown('**🔗 Sumber:**')
#             for src in msg['sources']:
#                 st.markdown(f'<span class="source-badge">🌐 {src}</span>', unsafe_allow_html=True)

# # ─────────────────────────────────────────────────────────────
# # CHAT INPUT
# # ─────────────────────────────────────────────────────────────
# user_input = st.chat_input('Tanyakan tentang perlindungan anak di ruang digital...')

# if user_input:
#     st.session_state.messages.append({'role': 'user', 'content': user_input})
#     with st.chat_message('user'):
#         st.markdown(user_input)

#     with st.chat_message('assistant'):
#         status_box = st.empty()

#         rag_context = ''
#         if use_rag and st.session_state.rag_built and 'rag_db' in st.session_state:
#             status_box.info('Mencari di dokumen regulasi...')
#             rag_context = retrieve_context(user_input, st.session_state.rag_db)

#         system_prompt = build_system_prompt(rag_context)
#         status_box.info('Menganalisis pertanyaan...')

#         tools_list = [exa_tool_decl] if use_exa else []

#         first_response = gemini_client.models.generate_content(
#             model='gemini-2.5-flash-lite',
#             contents=user_input,
#             config=gtypes.GenerateContentConfig(
#                 system_instruction=system_prompt,
#                 tools=tools_list if tools_list else None,
#             ),
#         )

#         candidate  = first_response.candidates[0]
#         parts      = candidate.content.parts
#         final_text = ''
#         sources    = []
#         tool_called = False

#         for part in parts:
#             if hasattr(part, 'function_call') and part.function_call:
#                 tool_called = True
#                 fn_name = part.function_call.name
#                 args    = dict(part.function_call.args)
#                 query   = args.get('query', user_input)
#                 status_box.info(f'Mencari di web: *{query}*')
#                 tool_result = exa_search(query)
#                 try:
#                     results_json = json.loads(tool_result)
#                     sources = [r['url'] for r in results_json if 'url' in r]
#                 except:
#                     pass
#                 status_box.info('Merangkum jawaban...')
#                 second_response = gemini_client.models.generate_content(
#                     model='gemini-2.5-flash-lite',
#                     contents=[
#                         user_input,
#                         gtypes.Content(role='model', parts=[part]),
#                         gtypes.Content(
#                             role='tool',
#                             parts=[
#                                 gtypes.Part.from_function_response(
#                                     name=fn_name,
#                                     response={'result': tool_result},
#                                 )
#                             ],
#                         ),
#                     ],
#                     config=gtypes.GenerateContentConfig(system_instruction=system_prompt),
#                 )
#                 final_text = second_response.text

#         if not tool_called:
#             final_text = first_response.text

#         status_box.empty()
#         st.markdown(final_text)

#         if sources:
#             st.markdown('**🔗 Sumber Web:**')
#             for url in sources:
#                 st.markdown(f'<span class="source-badge">🌐 {url}</span>', unsafe_allow_html=True)
#         if rag_context:
#             st.markdown('<span class="source-badge">📄 PP Tunas (Dokumen Lokal)</span>', unsafe_allow_html=True)

#     st.session_state.messages.append({
#         'role':    'assistant',
#         'content': final_text,
#         'sources': sources,
#     })


Overwriting streamlit_tunas_app.py


##  Sel 3 — Jalankan Streamlit via ngrok

In [29]:
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata

# Ambil ngrok token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

def run_streamlit(filename='streamlit_tunas_app.py', port=8501):
    # Hentikan proses streamlit & ngrok sebelumnya
    subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
    subprocess.run(['fuser', '-k', f'{port}/tcp'], capture_output=True)
    ngrok.kill()
    time.sleep(3)

    # Jalankan Streamlit
    proc = subprocess.Popen(
        [
            'streamlit', 'run', filename,
            '--server.headless=true',
            '--server.port', str(port),
            '--server.enableCORS=false',
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    time.sleep(4)

    # Buat tunnel publik
    public_url = ngrok.connect(port)
    print(f'\n🌱 TUNAS Digital berjalan di: {public_url}')
    print('Buka URL di atas di browser kamu!')
    return proc

proc = run_streamlit()


🌱 TUNAS Digital berjalan di: NgrokTunnel: "https://swear-stranger-churn.ngrok-free.dev" -> "http://localhost:8501"
Buka URL di atas di browser kamu!


## Sel 4 — Hentikan Aplikasi (Jalankan jika ingin berhenti)

In [ ]:
try:
    proc.terminate()
    print(' Streamlit dihentikan.')
except:
    print('Tidak ada proses yang berjalan.')

ngrok.kill()
print('Tunnel ngrok ditutup.')